## Running this notebook

Install the project with the notebook extra before opening this file. Imports
use the installed package and do not change the working directory. Set
`OVERSCREENING_OUTPUT_DIR` to choose a directory for saved files.

The original scientific parameter values and sweep sizes are retained. Several
cells are deliberately expensive or demonstrate a failed solve; they are not
part of the bounded CI examples. Inspect convergence flags before interpreting
results. Adjoint initialization and unsupported legacy boundary/operator
experiments are explicitly labelled and use the canonical collocation path.
For bounded NumPy/mpmath demonstrations, run the scripts in `examples/`.


## This notebook provides a set of examples on how to use the numerical method for  problems on a sement.

In [ ]:
# Install the project first: python -m pip install -e ".[notebook]"
from pathlib import Path
import os
from matplotlib import pyplot as plt
import numpy as np
import mpmath as mp
from overscreening_breakdown import solve, load_solution, overscreening_breakdown, overscreening_breakdown_segment
from overscreening_breakdown.arithmetic import MpmathArithmetic, NumpyArithmetic
from overscreening_breakdown.rational_mapping import RationalHalfLineMap
# The canonical legacy path is used only for the explicitly labelled adjoint or
# unsupported-operator/boundary experiments retained below.
from overscreening_breakdown.collocation.spectral1d import solve_nonlinear_problem
from overscreening_breakdown.collocation.adapter import CollocationProblemAdapter
from overscreening_breakdown.collocation.solution import CollocationSolution
OUTPUT_DIR = Path(os.environ.get("OVERSCREENING_OUTPUT_DIR", "notebook-output")).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
%matplotlib inline

def status(result):
    print("nonlinear_converged=", result.nonlinear_converged,
          "quadrature_converged=", result.quadrature_converged)

def display_values(function, coordinates):
    # Float conversion is confined to plot data.
    return np.array([float(function(value)) for value in coordinates])

def mapped_values(result, angles):
    a = result.arithmetic
    if result.domain[1] == a.inf:
        mapping = RationalHalfLineMap(a, L=result.L)
        coordinates = [mapping.from_angle(a.scalar(value)) for value in angles]
    else:
        left, right = result.domain
        coordinates = [(a.cos(a.scalar(value))*(right-left)+(right+left))/2 for value in angles]
    return display_values(result.u, coordinates)

def run_collocation(problem, N, *, L=1, tolerance="1e-10", method="newton",
                    max_iterations=50, globalization=True, initial=None,
                    use_mpmath=False, precision=100, adjoint=False):
    if not adjoint:
        if initial is not None and hasattr(initial, "coefficients"):
            initial = (initial.coefficients if initial.N == N
                       else lambda x, a, previous=initial: previous.u(x))
        return solve(problem, N, solver="collocation", L=L, tolerance=tolerance,
                     method=method, max_iterations=max_iterations,
                     use_globalization=globalization, initial_guess=initial,
                     use_mpmath=use_mpmath, precision=precision)
    # Adjoint initialization has no option in the common entry point. Preserve
    # that original experiment through the canonical legacy collocation API,
    # then capture its coefficients in an owned-arithmetic result for evaluation.
    a = MpmathArithmetic(precision) if use_mpmath else NumpyArithmetic()
    physical = problem.bind(a)
    previous_bits = mp.mp.prec
    try:
        legacy = solve_nonlinear_problem(
            N=N, problem=CollocationProblemAdapter(physical), L=L,
            tolerance=tolerance, use_method=method, total_iterations=max_iterations,
            use_globalization=globalization, use_adjoint_opimization=True,
            use_mpmath=use_mpmath, prec=precision)
        coefficients = legacy.solve_problem(None if initial is None else initial.coefficients)
        return CollocationSolution(physical=physical, coefficients=coefficients, N=N, L=L,
                                   method=method, converged=legacy.is_converged(),
                                   newton_result=legacy.get_newton_result())
    finally:
        mp.mp.prec = previous_bits


In [ ]:
problem = overscreening_breakdown_segment()

In [ ]:
#set up the problem.
#If a parameter is not given, then a default value is used instead. 
#See the source file of "overscreening_breakdown" for more details for defauls values

sigma = 20.0
mu = 0.36
gamma = 1
delta = 10.0
u0 = 1.0
u0xxx = 0

#use some function as initial guess for the root in the Newton's method.
def some_func(x):
    return u0*np.exp(-x)

#apply parameters
problem.set_parameters(sigma = sigma, mu = mu, gamma = gamma, delta = delta, u0 = u0, u0xxx=u0xxx, initial_guess_solution=some_func)

In [ ]:
N = 100
tolerance = 1.0e-10
use_method = "newton"
use_adjoint_opimization = False


In [ ]:
#pass the problem to the solver

# solver.set_problem(problem)
print("domain:", problem.get_domain()) #check, that we are loading a version with the finite segment domain
print("boundary:",problem.get_boundary_conditions()) #and with correct boundary conditions

In [ ]:
result = run_collocation(problem, N, tolerance=tolerance, method=use_method)
c = result.coefficients
status(result)


In [ ]:
# Canonical legacy path only for the adjoint-initialization comparison.
use_adjoint_opimization = True
result_ao = run_collocation(problem, N, tolerance=tolerance, method=use_method,
                            adjoint=use_adjoint_opimization)
c_ao = result_ao.coefficients
status(result_ao)


In [ ]:
t = np.arange(0, np.pi, 0.005)
plt.title(r"finite-domain solution on mapped $[0,\pi]$")
plt.plot(t, mapped_values(result, t), '.')
plt.plot(t, mapped_values(result_ao, t))
plt.legend(["no adjoint initialization", "adjoint initialization"])
plt.show()
x = np.arange(0.0, 100.0, 0.1)
# Evaluate the finite solution only on its actual [0,50] domain. Retain the
# original plotting grid and explicitly mask the points outside that domain.
inside = x <= float(result.domain[1])
plt.title("solution on its physical [0,50] domain")
plt.plot(x[inside], display_values(result.u, x[inside]), '.')
plt.plot(x[inside], display_values(result_ao.u, x[inside]))
plt.legend(["no adjoint initialization", "adjoint initialization"])
plt.show()


Let's check the inifinite domain variant and compare it side by side

In [ ]:
problem_inf = overscreening_breakdown()
sigma = 20.0
mu = 0.36
gamma = 1
delta = 10.0
u0 = 1.0
u0xxx = 0
def some_func(x):
    return u0*np.exp(-x)
problem_inf.set_parameters(sigma=sigma, mu=mu, gamma=gamma, delta=delta,
                            u0=u0, u0xxx=u0xxx, initial_guess_solution=some_func)


In [ ]:
N = 100
tolerance = 1.0e-10
use_method = "newton"
domain = [0,np.inf]
use_adjoint_opimization = True
L = 6
print("domain:", problem_inf.get_domain())
print("boundary:", problem_inf.get_boundary_conditions())


In [ ]:
# The physical description is passed directly to the solve call below.


In [ ]:
result_inf = run_collocation(problem_inf, N, L=L, tolerance=tolerance,
                              method=use_method, adjoint=use_adjoint_opimization)
c_inf = result_inf.coefficients
status(result_inf)


In [ ]:
x_max = 50
x = np.arange(0.0, x_max, x_max/1000.0)
plt.title("solution on 0 <= x <= {:.0f}".format(x_max))
plt.plot(x, display_values(result.u, x), '.')
plt.plot(x, display_values(result_inf.u, x))
plt.legend(["finite segment", r"$\mathbb{R}^+$"])
plt.show()


Let's solve some other problems on a segment...

In [ ]:
# to be continued